In [ ]:
!pip install langchain langchain-community langchain-core langchain-text-splitters langchain-anthropic sentence-transformers faiss-cpu pypdf --quiet

##Imports

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_anthropic import ChatAnthropic

## Split PDF

In [ ]:
pdf = PyPDFLoader("/content/Supply Chain PDF2.pdf")
document = pdf.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
Chunk = splitter.split_documents(document)

print(f"PDF loaded and split into {len(Chunk)} chunks")

## Embeddings and Vector Store

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.from_documents(documents=Chunk, embedding=embeddings)
retriever = db.as_retriever(search_kwargs={"k": 3})

print("Vector store ready")

## Load Anthropic

In [ ]:
from google.colab import userdata
ANTHROPIC_KEY = "API Key"
llm = ChatAnthropic(
    model="claude-haiku-4-5",
    api_key=ANTHROPIC_KEY
)
print("LLM ready")

## RAG Chain with memory

In [ ]:
chat_history = []

def ask(question):
    # Step 1: retrieve relevant chunks from PDF
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    # Step 2: build last 4 exchanges as history
    history_text = ""
    for role, msg in chat_history[-4:]:
        history_text += f"{role}: {msg}\n"

    # Step 3: build prompt
    full_prompt = f"""You are a helpful and friendly assistant who answers questions about a document.
Be conversational and natural — if someone greets you or chats casually, respond warmly.
For document questions, use the context provided.
If you don't know the answer from the document, say so honestly.

Previous conversation:
{history_text}

Relevant document context:
{context}

User: {question}
Assistant:"""

    # Step 4: call LLM
    result = llm.invoke(full_prompt)
    answer = result.content

    # Step 5: save to history
    chat_history.append(("User", question))
    chat_history.append(("Assistant", answer))

    return answer

print("RAG chain ready")

##Interactive Chat

In [ ]:
print("Chat with your PDF! Type 'quit' to exit or 'reset' to clear memory.\n")

while True:
    user_input = input("You: ").strip()

    if not user_input:
        continue
    elif user_input.lower() == "quit":
        print("Goodbye!")
        break
    elif user_input.lower() == "reset":
        chat_history = []
        print("Memory cleared!\n")
        continue

    response = ask(user_input)
    print(f"\nAssistant: {response}\n")